In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data_ESI_CorA =  "../data/original/Bonn/concentration/230203_DATA_Standort Bonn_Untersuchungen.xlsx"
data_AMELAG = "../data/original/Bonn/concentration/2024.02.29_AMELAG_Bonn Salierweg.xlsx"

#### Load Data and Unify Columns

In [ ]:
# load data
df_ESI_CorA = pd.read_excel(data_ESI_CorA)

# unify names and select relevant columns
df_ESI_CorA["Sampling_Area"] = df_ESI_CorA["PN Stelle"].apply(lambda x: "North" if x == "Zulauf Nord" else ("South" if x=="Zulauf Süd" else ("North_South" if x=="Misch Nord/Süd" else "Unknown")))
df_ESI_CorA.rename(columns={"Datum Probennahme\nBeginn [DD.MM.YYYY]": "Date", "Wasser-temperatur [°C]": "Temperature", "Mittlerer Durchfluss\n[l/s]":"Avg_FlowRate",
                   "SARS-CoV-2 N1": "COVID_N1", "SARS-CoV-2 N2 ": "COVID_N2","PMMoV ":"PMMoV",}, inplace=True)

df_ESI_CorA = df_ESI_CorA[["Sampling_Area", "Date",  "Avg_FlowRate", "COVID_N1", "COVID_N2", "PMMoV", "Temperature", "Bemerkung"]]

In [ ]:
# remove rows with missing values for COVID_N1 and COVID_N2
print(len(df_ESI_CorA))
df_ESI_CorA = df_ESI_CorA.loc[~(df_ESI_CorA.COVID_N1.isna() & df_ESI_CorA.COVID_N2.isna())].iloc[1:]    
print(len(df_ESI_CorA))

In [ ]:
# special observations
df_ESI_CorA.loc[df_ESI_CorA["Bemerkung"].isin(df_ESI_CorA["Bemerkung"].dropna().unique())]

In [ ]:
df_ESI_CorA.head()

In [ ]:
# same for AMELAG
df_AMELAG = pd.read_excel(data_AMELAG)
df_AMELAG.rename(columns={"Datum Start Probenahme": "Date", "mittlerer Durchfluss im Probenahmezeitraum": "Avg_FlowRate", "Temperatur": "Temperature", 
                          "PCR-Daten für SARS-CoV-2 N1":"COVID_N1", "PCR-Daten für SARS-CoV-2 N2": "COVID_N2", "PCR-Daten für PMMoV":"PMMoV", "Bemerkungen": "Bemerkung"}, inplace=True)
df_AMELAG["Sampling_Area"] = "North_South"
df_AMELAG = df_AMELAG[["Sampling_Area", "Date", "Avg_FlowRate", "COVID_N1", "COVID_N2", "PMMoV", "Temperature", "Bemerkung"]]
df_AMELAG = df_AMELAG.loc[df_AMELAG.Date<="2024-02-29"]  # filter out future dates
# fillna values with zeros
df_AMELAG.fillna({"COVID_N1": 0, "COVID_N2": 0}, inplace=True)

### Combine Data

In [ ]:
df_AMELAG["Project"] = "AMELAG"
df_ESI_CorA["Project"] = "ESI_CorA"

df = pd.concat([df_ESI_CorA, df_AMELAG], axis=0)
df.sort_values(by="Date", inplace=True)

Column description

* Sampling Area = PN Stelle, one of *North*, *South* or *North_South*
* Date = Date of Sampling (Datum Probenentnahme Beginn)
* Avg_FlowRate = Average Flow Rate during sampling period in [l/s]
* COVID_N1, COVID_N2, PMMoV in [copies/l]
* Temperature = Water temperature when taking the sample [°C]
* Bemerkung = further information 
* Project one of ESI_CorA or AMELAG

#### Calculate North_South values for missing timepoints

In [ ]:
df_sub = df.loc[~df.Date.isin(df.loc[df.Sampling_Area=="North_South"].Date.unique()),:]
if len(df_sub.Project.unique()) > 1:
    raise ValueError("Data contains multiple projects, which is not expected.")
df_sub = df_sub.pivot_table(index=["Date"], columns=["Sampling_Area"], values=["COVID_N1", "COVID_N2", "PMMoV", "Avg_FlowRate"], aggfunc=np.mean).reset_index()

In [ ]:
# gewichtete Mittelwerte für Mischproben
gesamtdurchfluss = df_sub[('Avg_FlowRate', 'North')] + df_sub[('Avg_FlowRate', 'South')]

SARS_CoV_N1 = df_sub[('COVID_N1', 'North')]*df_sub[('Avg_FlowRate', 'North')] / gesamtdurchfluss + df_sub[('COVID_N1', 'South')]*df_sub[('Avg_FlowRate', 'South')] / gesamtdurchfluss
SARS_CoV_N2 = df_sub[('COVID_N2', 'North')]*df_sub[('Avg_FlowRate', 'North')] / gesamtdurchfluss + df_sub[('COVID_N2', 'South')]*df_sub[('Avg_FlowRate', 'South')] / gesamtdurchfluss

PMMoV = df_sub[('PMMoV', 'North')]*df_sub[('Avg_FlowRate', 'North')] / gesamtdurchfluss + df_sub[('PMMoV', 'South')]*df_sub[('Avg_FlowRate', 'South')] / gesamtdurchfluss

In [ ]:
df_new_dates = pd.DataFrame(df_sub["Date"])
df_new_dates["Sampling_Area"] = "North_South"
df_new_dates["COVID_N1"] = SARS_CoV_N1
df_new_dates["COVID_N2"] = SARS_CoV_N2
df_new_dates["PMMoV"] = PMMoV
df_new_dates["Avg_FlowRate"] = df_sub[('Avg_FlowRate', 'North')] + df_sub[('Avg_FlowRate', 'South')]
df_new_dates["Project"] = "ESI_CorA"

In [ ]:
df = pd.concat([df_new_dates, df], axis=0)

In [ ]:
df["COVID_N1"]

### Apply "Bestimmungsgrenze" and drop rows

In [ ]:
df["bestimmungsgrenze"] = df.apply(lambda x: True if ((x["COVID_N1"]<=1000) or (x["COVID_N2"]<=1000) or (x["PMMoV"]<=1000)) else False, axis=1)
for col in ["COVID_N1", "COVID_N2", "PMMoV"]:
    df[col] = df[col].apply(lambda x: x/2 if x==1000 else x)

In [ ]:
df[df["bestimmungsgrenze"]]

In [ ]:
df = df.loc[df.bestimmungsgrenze==False]    

### get weather conditions

In [ ]:
starkregengrenze = 76000*1000/(24*60*60)
df["precipitation_event"] = df["Avg_FlowRate"].apply(lambda x: None if x==None else "light_rain") # Leichter Regen
df.loc[df["Avg_FlowRate"]>starkregengrenze, "precipitation_event"] = "heavy_rain" # Starkregen
trockenregengrenze = 40000*1000/(24*60*60)
df.loc[df["Avg_FlowRate"]<trockenregengrenze, "precipitation_event"] = "dry" # Trockenwetter

In [ ]:
df["Avg_FlowRate"].isna().sum()

In [ ]:
df["precipitation_event"].value_counts(normalize=True, dropna=False)

### apply flow and PMMoV normalization

In [ ]:
df.columns

In [ ]:
df.loc[df["precipitation_event"]=="dry", "Avg_FlowRate"].mean()

In [ ]:
# flow normalization
# [COV]*flow_rate/avg_dry_weather_flow_rate
avg_dry_weather_flow_rate = df.loc[df["precipitation_event"]=="dry", "Avg_FlowRate"].mean()
for col in ["COVID_N1", "COVID_N2"]:
    new_col_name = "flow_normalized_"+col
    df[new_col_name] = df[col]*df["Avg_FlowRate"]/avg_dry_weather_flow_rate

In [ ]:
# PMMoV normalization
# [COV]*avg_dry_weather_PMMoV_value/[PMMoV]
avg_dry_weather_PMMoV_value = df.loc[df["precipitation_event"]=="dry", "PMMoV"].mean()
for col in ["COVID_N1", "COVID_N2"]:
    new_col_name = "PMMoV_normalized_" + col
    df[new_col_name] = df[col]*avg_dry_weather_PMMoV_value/df["PMMoV"]

#### store results

In [ ]:
# tidy temperature columns
df.loc[df.Temperature ==" -", "Temperature"] = np.nan
df.Temperature = df.Temperature.astype(float)

In [ ]:
df.to_csv("../data/preprocessed/wastewater.csv", index=False)